In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_squared_error, r2_score
import joblib
from sklearn.ensemble import RandomForestRegressor

# read data
data = pd.read_csv("F:/Jupyter/student project/zs/zs3 20250603.csv")
X = data[data.columns[3:]]
y = data['APACHEII']
  
#train and test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=99)

# Save the group data
train = pd.concat([y_train, X_train], axis=1)
train.to_csv("F:/Jupyter/student project/zs/train_feature 3.csv", index=False)
test = pd.concat([y_test, X_test], axis=1)
test.to_csv("F:/Jupyter/student project/zs/test_feature 3.csv", index=False)

# feature selection： 
selector = RFE(RandomForestRegressor(random_state=42), n_features_to_select=20)
selector.fit(X_train, y_train)

selected_mask = selector.get_support()
X_train_sel = X_train.loc[:, selected_mask]
X_test_sel = X_test.loc[:, selected_mask]

print("train shape:", X_train_sel.shape)
print("test shape:", X_test_sel.shape)

# ---------- grid search ----------
param_grid = {
    'n_estimators': [50, 80, 100, 150],
    'max_depth': [5, 8, 10, None],
    'min_samples_split': [2, 3, 5],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.6, 0.8]    
}

rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(estimator=rf,
                           param_grid=param_grid,
                           scoring='neg_mean_squared_error',
                           cv=5,
                           n_jobs=-1,
                           verbose=1)

# search on train set
grid_search.fit(X_train_sel, y_train)

# best parameters
print("best parameters:", grid_search.best_params_)

# best model
best_rf = grid_search.best_estimator_
joblib.dump(best_rf, 'random_forest_model.pkl')

# ---------- evaluation ----------
y_train_pred = best_rf.predict(X_train_sel)
y_test_pred = best_rf.predict(X_test_sel)

print("\n--- train performance ---")
print("MSE: {:.3f}".format(mean_squared_error(y_train, y_train_pred)))
print("R² : {:.3f}".format(r2_score(y_train, y_train_pred)))

print("\n--- test performance ---")
print("MSE: {:.3f}".format(mean_squared_error(y_test, y_test_pred)))
print("R² : {:.3f}".format(r2_score(y_test, y_test_pred)))

# ---------- mse----------
fig, ax = plt.subplots(figsize=(8,6),dpi = 300)
plt.scatter(y_train_pred,
            y_train_pred - y_train,
            c='brown',
            edgecolor='white',
            marker='o',
            s=50,
            alpha=0.9,
            label=f'train: MSE({train_mse:.3f}) R²({train_r2:.3f})')
plt.scatter(y_test_pred,
            y_test_pred - y_test,
            c='navy', edgecolor='white',    marker='D',
            s=50,
            alpha=0.9,
            label=f'test: MSE({test_mse:.3f}) R²({test_r2:.3f})')

plt.xlabel('Predicted values')
plt.ylabel('Residuals')
plt.legend(loc='upper right')
plt.hlines(y=0, xmin=5, xmax=25, lw=2, linestyle='--',color='black')
plt.xlim([5, 25])
plt.tight_layout()
plt.savefig("F:/Jupyter/student project/zs/MSE 20.png")


# ---------- calibration----------
def calibration_metrics(y_true, y_pred, n_bins=10):
    """
    Calculate calibration metrics based on equal-frequency binning:
    slope, intercept, R², and calibration-in-the-large. 
    """
    # Equal-frequency binning based on predicted values
    bin_edges = np.percentile(y_pred, np.linspace(0, 100, n_bins + 1))
    bin_indices = np.digitize(y_pred, bin_edges[:-1])

    bin_true_mean = []
    bin_pred_mean = []
    for i in range(1, n_bins + 1):
        mask = bin_indices == i
        if mask.sum() > 0:
            bin_true_mean.append(np.mean(y_true[mask]))
            bin_pred_mean.append(np.mean(y_pred[mask]))

    # Convert to arrays
    bin_pred_mean = np.array(bin_pred_mean)
    bin_true_mean = np.array(bin_true_mean)

    # Linear regression: observed means ~ predicted means
    slope, intercept, r_value, p_value, std_err = stats.linregress(bin_pred_mean, bin_true_mean)

    # Calibration-in-the-large
    cal_large = np.mean(y_pred) - np.mean(y_true)

    metrics = {
        'slope': slope,
        'intercept': intercept,
        'r_squared': r_value**2,
        'calibration_in_large': cal_large,
        'mean_obs': np.mean(y_true),
        'mean_pred': np.mean(y_pred)
    }

    bin_df = pd.DataFrame({
        'bin_pred_mean': bin_pred_mean,
        'bin_true_mean': bin_true_mean
    })

    return metrics, bin_df


def plot_calibration_curve(y_true, y_pred, metrics=None, n_bins=10, 
                           title="Calibration Plot", save_path=None):
    # Compute bin means (reuse calibration_metrics logic to avoid recomputation if metrics is given)
    bin_edges = np.percentile(y_pred, np.linspace(0, 100, n_bins + 1))
    bin_indices = np.digitize(y_pred, bin_edges[:-1])

    bin_true_mean = []
    bin_pred_mean = []
    for i in range(1, n_bins + 1):
        mask = bin_indices == i
        if mask.sum() > 0:
            bin_true_mean.append(np.mean(y_true[mask]))
            bin_pred_mean.append(np.mean(y_pred[mask]))

    bin_pred_mean = np.array(bin_pred_mean)
    bin_true_mean = np.array(bin_true_mean)

    plt.figure(figsize=(6, 6))
        
    # Plot binned points
    plt.plot(bin_pred_mean, bin_true_mean, 'o-', label='Binned means', markersize=8)

    # Perfect calibration line (y=x)
    min_val = min(min(bin_pred_mean), min(bin_true_mean))
    max_val = max(max(bin_pred_mean), max(bin_true_mean))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect calibration')

    plt.xlabel('Mean Predicted Value')
    plt.ylabel('Mean Observed Value')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()


# --- Compute metrics for training and test sets ---
train_metrics, train_bin = calibration_metrics(y_train, y_train_pred, n_bins=10)
test_metrics, test_bin = calibration_metrics(y_test, y_test_pred, n_bins=10)

# --- Print metrics ---
print("===== Training Set Calibration Metrics =====")
print(f"Slope: {train_metrics['slope']:.3f} (ideal=1)")
print(f"Intercept: {train_metrics['intercept']:.3f} (ideal=0)")
print(f"R²: {train_metrics['r_squared']:.3f}")
print(f"Calibration-in-the-large: {train_metrics['calibration_in_large']:.3f}")
print(f"Mean observed: {train_metrics['mean_obs']:.2f}, Mean predicted: {train_metrics['mean_pred']:.2f}")

print("\n===== Test Set Calibration Metrics =====")
print(f"Slope: {test_metrics['slope']:.3f} (ideal=1)")
print(f"Intercept: {test_metrics['intercept']:.3f} (ideal=0)")
print(f"R²: {test_metrics['r_squared']:.3f}")
print(f"Calibration-in-the-large: {test_metrics['calibration_in_large']:.3f}")
print(f"Mean observed: {test_metrics['mean_obs']:.2f}, Mean predicted: {test_metrics['mean_pred']:.2f}")

# --- Plot calibration curves ---
plot_calibration_curve(y_train, y_train_pred, metrics=train_metrics, 
                       title="Training Set Calibration", save_path="F:/Jupyter/student project/zs/train_cal_20.png")
plot_calibration_curve(y_test, y_test_pred, metrics=test_metrics, 
                       title="Test Set Calibration", save_path="F:/Jupyter/student project/zs/test_cal_20.png")

FileNotFoundError: [Errno 2] No such file or directory: 'F:/Jupyter/student project/zs/zs3 20250603.csv'